# Feeding the CPU data into a GPU

### Importing packages

In [ ]:
%%capture
%pip install --upgrade nvidia-cuda-nvcc-cu12
%pip install --upgrade "jax[cuda12]" numpyro

In [ ]:
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO, autoguide
from jax import random
from jax.scipy.special import logsumexp

### Simulating raw data

In [ ]:
print("Generating synthetic tick data...")
n_ticks = 500_000 # Half a million ticks

prices = 1.1000 + np.cumsum(np.random.normal(0, 0.0001, n_ticks))
volumes = np.random.randint(10_000, 500_000, n_ticks) # Volume per tick
timestamps = pd.date_range("2019-01-01", periods=n_ticks, freq="1S")

tick_df = pd.DataFrame({'price': prices, 'volume': volumes}, index=timestamps)
print(tick_df)

### Compress to volume bars

In [ ]:
def create_volume_bars(df, volume_threshold=5_000_000):
    print(f"compressing ticks into {volume_threshold:,} volume bars...")

    cumulative_vol = df['volume'].cumsum()

    bar_ids = cumulative_vol // volume_threshold

    bars = df.groupby(bar_ids).agg(
            timestamp=('price', lambda x: x.index[-1]), # End time of the bar
            open=('price', 'first'),
            high=('price', 'max'),
            low=('price', 'min'),
            close=('price', 'last'),
            volume=('volume', 'sum')
        )

    bars.set_index('timestamp', inplace=True)

    bars['returns'] = bars['close'].pct_change().fillna(0)

    return bars

volume_bars = create_volume_bars(tick_df)
print(f"Reduced {len(tick_df)} ticks to {len(volume_bars)} Volume Bars.")

### JAX/NumPyro GPU Model and "Chunking"

In [ ]:
chunk_size = 100
n_chunks = len(volume_bars) // chunk_size

matrix_data = volume_bars['returns'].values[:n_chunks * chunk_size]

y_matrix = matrix_data.reshape(n_chunks, chunk_size)

y_jax = jnp.array(y_matrix)
print(f"data pushed to GPU with matrix shape: {y_jax.shape} (chunks, bars)")


### Define the HMM for the GPU using NumPyro

In [ ]:
def hmm_model(y):
    n_days, n_bars = y.shape
    n_states = 2

    # 1. Priors for the Transition Matrix
    transition_probs = numpyro.sample(
        "transition_probs", 
        dist.Dirichlet(jnp.ones((n_states, n_states)))
    )

    # 2. Priors for State Parameters (Mean and Variance)
    mu = numpyro.sample("mu", dist.Normal(jnp.zeros(n_states), 0.01))
    sigma = numpyro.sample("sigma", dist.LogNormal(jnp.zeros(n_states), 1.0))
    init_probs = numpyro.sample("init_probs", dist.Dirichlet(jnp.ones(n_states)))

    # 3. The Forward Algorithm (Manual HMM Math)
    # We define how to calculate the log-likelihood of a single day of returns
    def forward_one_day(y_day):
        
        # Step A: Initialize the first bar of the day
        log_alpha_init = jnp.log(init_probs) + dist.Normal(mu, sigma).log_prob(y_day[0])
        
        # Step B: Loop through the rest of the day chronologically using jax.lax.scan
        def scan_fn(log_alpha_prev, y_t):
            log_P = jnp.log(transition_probs)
            log_emission = dist.Normal(mu, sigma).log_prob(y_t)
            
            # The core HMM equation: log(alpha_t) = logsumexp(alpha_{t-1} * P) * emission
            # We use logsumexp to prevent floating-point underflow on the GPU
            log_alpha_next = logsumexp(log_alpha_prev[:, None] + log_P, axis=0) + log_emission
            
            # Return the new state to carry forward, and 'None' for accumulated memory
            return log_alpha_next, None 
            
        # Run the fast GPU loop over the remaining bars (y_day[1:])
        log_alpha_final, _ = jax.lax.scan(scan_fn, log_alpha_init, y_day[1:])
        
        # Total log-likelihood of the day is the sum of probabilities of ending in any state
        return logsumexp(log_alpha_final)
        
    # 4. Vectorize across ALL days simultaneously!
    # jax.vmap takes our single-day function and maps it across the 
    # n_days matrix dimension so the H100 calculates every day concurrently.
    log_likes = jax.vmap(forward_one_day)(y)
    
    # 5. Tell NumPyro to optimize based on the sum of all daily log-likelihoods
    numpyro.factor("log_likelihood", jnp.sum(log_likes))

### Training the model

In [ ]:
print("\nCompiling model to GPU...")
optimizer = numpyro.optim.Adam(step_size=0.01)

# The "guide" acts as Maximum Likelihood Estimation
guide = autoguide.AutoDelta(hmm_model)
svi = SVI(hmm_model, guide, optimizer, loss=Trace_ELBO())

rng_key = random.PRNGKey(42)

# Initialize the SVI state
svi_state = svi.init(rng_key, y=y_jax)

n_steps = 1000
losses = [] # List to store our loss at every step

print("Starting training loop...")

# Run the training loop explicitly to capture progress
for i in range(n_steps):
    svi_state, loss = svi.update(svi_state, y=y_jax)
    # Ensure we pull the loss value out of the JAX array format
    losses.append(float(loss)) 
    
    # Just print text updates along the way
    if i % 100 == 0 or i == n_steps - 1:
        print(f"Step {i:4d} | Loss (Negative ELBO): {loss:.2f}")

print("\nTraining Complete!")

In [ ]:
# ==============================================================================
# POST-TRAINING PLOT
# ==============================================================================
# Convert the list of losses into a Pandas DataFrame for easy handling
loss_df = pd.DataFrame({'Negative_ELBO': losses})

plt.figure(figsize=(10, 5))
plt.plot(loss_df.index, loss_df['Negative_ELBO'], color='royalblue', linewidth=2, label='Training Loss')
plt.title("SVI GPU Training Convergence")
plt.xlabel("Iteration Step")
plt.ylabel("Loss (Negative ELBO)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ==============================================================================
# EXTRACT PARAMETERS
# ==============================================================================
learned_params = svi.get_params(svi_state)

print("\n--- GPU Learned Parameters ---")
print("Transition Matrix:\n", np.round(learned_params['transition_probs_auto_loc'], 4))
print("State Volatilities (Sigma):\n", np.round(learned_params['sigma_auto_loc'], 5))